In [0]:
"""
Cell 1: FeatureEngineer
OOP concept: har feature-computation method chhota aur single-purpose hai
(Single Responsibility), aur ek master method (`transform`) sabko chain karta hai.
"""

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window


class FeatureEngineer:
    """Silver transactions se ML-ready features banata hai."""

    def add_balance_features(self, df: DataFrame) -> DataFrame:
        return (
            df
            .withColumn(
                "orig_balance_ratio",
                F.when(F.col("oldbalanceOrg") > 0,
                       F.col("amount") / F.col("oldbalanceOrg"))
                 .otherwise(0.0)
            )
            .withColumn(
                "orig_drained_to_zero",
                F.when(
                    (F.col("oldbalanceOrg") > 0) & (F.col("newbalanceOrig") == 0),
                    1
                ).otherwise(0)
            )
            .withColumn(
                "dest_balance_ratio",
                F.when(F.col("oldbalanceDest") > 0,
                       F.col("amount") / F.col("oldbalanceDest"))
                 .otherwise(0.0)
            )
        )

    def add_velocity_features(self, df: DataFrame) -> DataFrame:
        """
        Velocity = same account (nameOrig) ne kitni transactions ki hain
        ab tak (running count) — fraud mein aksar burst hota hai.
        """
        window_spec = (
            Window.partitionBy("nameOrig")
            .orderBy("step")
            .rowsBetween(Window.unboundedPreceding, Window.currentRow)
        )
        return df.withColumn(
            "orig_txn_count_so_far",
            F.count("transaction_id" if "transaction_id" in df.columns else "step").over(window_spec)
        )

    def add_type_encoding(self, df: DataFrame) -> DataFrame:
        """Transaction type ko one-hot style flags mein convert karna."""
        return (
            df
            .withColumn("is_cash_out", (F.col("type") == "CASH_OUT").cast("int"))
            .withColumn("is_transfer", (F.col("type") == "TRANSFER").cast("int"))
            .withColumn("is_payment", (F.col("type") == "PAYMENT").cast("int"))
        )

    def transform(self, df: DataFrame) -> DataFrame:
        """Master method — sab feature steps ko sahi order mein chain karta hai."""
        df = self.add_balance_features(df)
        df = self.add_velocity_features(df)
        df = self.add_type_encoding(df)
        return df


feature_engineer = FeatureEngineer()
print(" FeatureEngineer ready")

In [0]:
silver_df = spark.table("workspace.default.silver_transactions")

feature_df = feature_engineer.transform(silver_df)

(
    feature_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.feature_transactions")
)

print(" Feature table saved")
spark.sql("SELECT COUNT(*) FROM workspace.default.feature_transactions").show()

In [0]:


from pyspark.sql import functions as F
from pyspark.sql.window import Window


class VelocityFeatureEngineer:
    """Time-based sliding-window features — fraud ka sabse strong signal."""

    def add_synthetic_event_time(self, df):
        """step (hour) ke andar random second assign karna, real-time simulate karne ke liye."""
        base_time = F.lit("2024-01-01 00:00:00").cast("timestamp")
        return df.withColumn(
            "event_time",
            base_time
            + (F.col("step") * 3600).cast("long").cast("interval second")
            + (F.rand(seed=42) * 3600).cast("long").cast("interval second")
        ).withColumn(
            "event_time_unix", F.col("event_time").cast("long")
        )

    def add_rolling_windows(self, df):
        """
        Har account (nameOrig) ke liye: pichle 1 min, 5 min, 15 min mein
        kitni transactions hui aur total kitna amount gaya — sliding window
        with rangeBetween (time-based, row-based nahi).
        """
        def window(seconds):
            return (
                Window.partitionBy("nameOrig")
                .orderBy("event_time_unix")
                .rangeBetween(-seconds, 0)
            )

        df = df.withColumn("txn_count_1min", F.count("amount").over(window(60)))
        df = df.withColumn("txn_count_5min", F.count("amount").over(window(300)))
        df = df.withColumn("txn_count_15min", F.count("amount").over(window(900)))
        df = df.withColumn("txn_amount_sum_1min", F.sum("amount").over(window(60)))
        df = df.withColumn("txn_amount_sum_5min", F.sum("amount").over(window(300)))
        return df

    def add_time_since_last_txn(self, df):
        """Isi account ki pichli transaction se kitna time guzra (seconds)."""
        w = Window.partitionBy("nameOrig").orderBy("event_time_unix")
        return df.withColumn(
            "seconds_since_last_txn",
            F.col("event_time_unix") - F.lag("event_time_unix", 1).over(w)
        ).fillna({"seconds_since_last_txn": 999999})

    def add_distinct_merchants_24h(self, df):
        """Pichle 24 ghante mein kitne alag merchants (nameDest) se transact kiya."""
        w = (
            Window.partitionBy("nameOrig")
            .orderBy("event_time_unix")
            .rangeBetween(-86400, 0)
        )
        return df.withColumn(
            "distinct_merchants_24h", F.approx_count_distinct("nameDest").over(w)
        )

    def transform(self, df):
        df = self.add_synthetic_event_time(df)
        df = self.add_rolling_windows(df)
        df = self.add_time_since_last_txn(df)
        df = self.add_distinct_merchants_24h(df)
        return df


velocity_engineer = VelocityFeatureEngineer()
print(" VelocityFeatureEngineer ready")

In [0]:
feature_df_v2 = velocity_engineer.transform(feature_df)

(
    feature_df_v2.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.feature_transactions")
)

print(" Velocity features added to feature table")
spark.sql("SELECT nameOrig, event_time, txn_count_5min, txn_amount_sum_5min, seconds_since_last_txn, distinct_merchants_24h FROM workspace.default.feature_transactions ORDER BY event_time LIMIT 10").show(truncate=False)